# Regresión Logística: Predicción de Diabetes

Notebook de **integrante2** para explorar el dataset `data/diabetes.csv`, implementar **Regresión Logística desde cero con NumPy** y compararla con `scikit-learn`.

## Pasos
1. Carga y exploración de datos (EDA)
2. División Train/Test y escalado sin fuga de datos (*data leakage*)
3. Implementación desde cero con NumPy (descenso de gradiente + *Binary Cross-Entropy*)
4. Implementación con `scikit-learn` (`LogisticRegression`)
5. Evaluación comparativa y conclusiones

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

## 1. Carga y exploración de datos

In [ ]:
df = pd.read_csv("../../data/diabetes.csv")
print("Dimensiones:", df.shape)
df.head()

In [ ]:
print("Valores nulos por columna:")
print(df.isna().sum())

print("\nDistribución de la variable objetivo (Resultado):")
print(df["Resultado"].value_counts(normalize=True).round(4))

df.describe().T

In [ ]:
# Revisar si existen ceros atípicos en mediciones fisiológicas
cols_fisiologicas = ["Glucosa", "PresionSanguinea", "EspesorPiel", "Insulina", "IMC"]
(df[cols_fisiologicas] == 0).sum()

## 2. División Train/Test y escalado

Separamos `80%` entrenamiento y `20%` prueba usando `stratify=y` para mantener la proporción de clases. El `StandardScaler` se ajusta **únicamente** con `X_train` para evitar fuga de datos.

In [ ]:
X = df.drop(columns=["Resultado"]).to_numpy(dtype=float)
y = df["Resultado"].to_numpy(dtype=int)
feature_names = df.drop(columns=["Resultado"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)

## 3. Regresión Logística desde cero con NumPy

Modelamos la probabilidad $\hat{p} = \sigma(Xw + b)$ donde $\sigma(z) = \frac{1}{1 + e^{-z}}$, minimizando la pérdida *Binary Cross-Entropy* mediante descenso de gradiente.

In [ ]:
class RegresionLogisticaNumPy:
    def __init__(self, lr: float = 0.1, n_iter: int = 1500) -> None:
        self.lr = lr
        self.n_iter = n_iter
        self.weights: np.ndarray | None = None
        self.bias: float = 0.0
        self.loss_history: list[float] = []

    @staticmethod
    def _sigmoid(z: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500.0, 500.0)))

    @staticmethod
    def _bce_loss(y_true: np.ndarray, probs: np.ndarray) -> float:
        eps = 1e-15
        p = np.clip(probs, eps, 1.0 - eps)
        return float(-np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p)))

    def fit(self, X: np.ndarray, y: np.ndarray) -> "RegresionLogisticaNumPy":
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.n_iter):
            probs = self._sigmoid(X @ self.weights + self.bias)
            self.loss_history.append(self._bce_loss(y, probs))

            error = probs - y
            dw = (1.0 / n_samples) * (X.T @ error)
            db = (1.0 / n_samples) * float(np.sum(error))
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        return (self.predict_proba(X) >= threshold).astype(int)


modelo_np = RegresionLogisticaNumPy(lr=0.1, n_iter=1500).fit(X_train_scaled, y_train)
pred_np = modelo_np.predict(X_test_scaled)

plt.figure(figsize=(7, 4))
plt.plot(modelo_np.loss_history)
plt.title("Curva de pérdida (Binary Cross-Entropy) - NumPy")
plt.xlabel("Iteración")
plt.ylabel("Log-Loss")
plt.grid(alpha=0.3)
plt.show()

## 4. Implementación con `scikit-learn`

In [ ]:
modelo_sk = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
modelo_sk.fit(X_train_scaled, y_train)
pred_sk = modelo_sk.predict(X_test_scaled)

## 5. Evaluación comparativa

In [ ]:
def evaluar(nombre: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }


resultados = pd.DataFrame(
    [
        evaluar("NumPy (GD)", y_test, pred_np),
        evaluar("scikit-learn", y_test, pred_sk),
    ]
)
resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (titulo, pred) in zip(
    axes, [("NumPy (GD)", pred_np), ("scikit-learn", pred_sk)]
):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"Matriz de confusión - {titulo}")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
coef_df = pd.DataFrame(
    {
        "Variable": feature_names,
        "Peso_NumPy": modelo_np.weights,
        "Peso_sklearn": modelo_sk.coef_[0],
    }
)
coef_df

### 5.1 Experimento: Imputación de ceros fisiológicos

Reemplazamos los `0` en `Glucosa`, `PresionSanguinea` e `IMC` por la mediana calculada **únicamente sobre `X_train`** (`SimpleImputer`) antes de escalar, y comparamos con el modelo base.

In [ ]:
from sklearn.impute import SimpleImputer

cols_a_imputar = [
    feature_names.index(c) for c in ["Glucosa", "PresionSanguinea", "IMC"]
]

X_train_imp = X_train.copy()
X_test_imp = X_test.copy()

imputer = SimpleImputer(missing_values=0, strategy="median")
X_train_imp[:, cols_a_imputar] = imputer.fit_transform(X_train_imp[:, cols_a_imputar])
X_test_imp[:, cols_a_imputar] = imputer.transform(X_test_imp[:, cols_a_imputar])

scaler_imp = StandardScaler()
X_train_imp_sc = scaler_imp.fit_transform(X_train_imp)
X_test_imp_sc = scaler_imp.transform(X_test_imp)

modelo_sk_imp = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
modelo_sk_imp.fit(X_train_imp_sc, y_train)
pred_sk_imp = modelo_sk_imp.predict(X_test_imp_sc)

comparacion_imp = pd.DataFrame(
    [
        evaluar("scikit-learn (sin imputar)", y_test, pred_sk),
        evaluar("scikit-learn (imputando ceros)", y_test, pred_sk_imp),
    ]
)
comparacion_imp

## 6. Conclusiones

- **NumPy vs. `scikit-learn`**: Ambos convergen a pesos y métricas muy similares. `scikit-learn` aplica regularización $L_2$ por defecto (`C=1.0`) y un optimizador cuasi-Newton (`lbfgs`), mientras que nuestra versión en NumPy usa descenso de gradiente simple sin regularización.
- **Observación sobre el dataset**: `Glucosa` e `IMC` suelen tener los pesos positivos más altos.